# F1 · No penalty, L2, L1 and elastic net with few true features and many useless ones

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Compare four regularizations of the logistic (none, L2, L1, elastic net) on 5 simulated datasets with 100 features of which only **6 informative** and 200 training examples, choosing `C` with
cross-validation **for each model**. Measure log-loss and AUC on 2000 new examples, how many features each model keeps and how many of the six true ones it finds again.

## Dataset

`generate(seed)` (given): 2200 examples, 100 features (the first 6 informative, the other 94 noise), 3% of labels swapped; training = the first 200, test = the other 2000. The features are standardized inside the `Pipeline`.

## Educational purpose

Answer with numbers "when do you need L1 and when L2?". With few true features among many useless ones the **sparse** penalty (L1, elastic net) has an advantage, not only in interpretability but in prediction
error; with no penalty at all the model is unusable.

## Task

1. Build `models`, a dictionary name → `Pipeline` (`StandardScaler` + `LogisticRegression`): `none` (`C=1e6, max_iter=20000`), `L2` (`solver="lbfgs"`), `L1` (`penalty="l1", solver="liblinear"`) and `elasticnet` (`penalty="elasticnet", solver="saga", l1_ratio=0.5`). All with `max_iter=5000` except `none`.
2. For `L2`, `L1` and `elasticnet` wrap the pipeline in `GridSearchCV` over `C` in `np.logspace(-3, 1, 9)` with `StratifiedKFold(5, shuffle=True, random_state=0)` and `scoring="neg_log_loss"`. `none` stays without a search.
3. For each seed from 0 to 4: generate the data, train the four models and record on the test set `logloss` (with probabilities clipped to `[1e-15, 1−1e-15]`), `auc`, `n_coef` (coefficients with absolute value > 1e-8) and `n_true` (how many of these have an index below 6, that is are among the informative features). Save in `table` the **mean over the 5 seeds** (rows: models).
4. Save `sparse_win` (True if the log-loss of `L1` **and** `elasticnet` are lower than that of `L2`) and `l1_true` (mean of the true features found by L1).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate(seed):
    X, y = make_classification(n_samples=2200, n_features=100, n_informative=6, n_redundant=0, class_sep=1.0, flip_y=0.03, shuffle=False, random_state=seed)
    o = np.random.default_rng(seed).permutation(len(y))         # shuffle the rows (the informative columns stay the first 6)
    X, y = X[o], y[o]
    return X[:200], y[:200], X[200:], y[200:]


cv = StratifiedKFold(5, shuffle=True, random_state=0)
Cs = np.logspace(-3, 1, 9)

## Your work

In [ ]:
def new_models():
    """Returns a dictionary name -> estimator, to be recreated for every seed."""
    ...

In [ ]:
rows = []
for seed in range(5):
    X_tr, y_tr, X_te, y_te = generate(seed)
    for name, m in new_models().items():
        ...
table = pd.DataFrame(rows).groupby("model").mean().drop(columns="seed")
table.round(3)

In [ ]:
sparse_win = None
l1_true = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`elasticnet`: `LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=5000)`. Search: `GridSearchCV(pipe, {"logisticregression__C": Cs}, cv=cv, scoring="neg_log_loss")`; after `fit` the best model is `search.best_estimator_[-1].coef_[0]`. `none` has no `best_estimator_`: use `m[-1].coef_[0]`.'):
    check(set(table.index) == {"none", "L2", "L1", "elasticnet"} and list(table.columns) == ["logloss", "auc", "n_coef", "n_true"], "table with four models and four metrics", "columns: logloss, auc, n_coef, n_true")
    check(table.loc["none", "logloss"] > 3 and table.loc["none", "n_coef"] == 100, "without penalty the test log-loss exceeds 3 and all 100 coefficients are nonzero", "C = 1e6")
    check(table.loc["L2", "n_coef"] == 100 and table.loc["L1", "n_coef"] < 50 and table.loc["elasticnet", "n_coef"] < 60, "L2 zeroes nothing; L1 and elastic net keep fewer than half of the features", "count |coef| > 1e-8")
    check(sparse_win is True and table.loc["L1", "auc"] > table.loc["L2", "auc"] + 0.03, "L1 and elastic net beat L2 on log-loss, and L1 by over 0.03 of AUC", "compare the logloss and auc columns")
    check(table.loc["L2", "logloss"] < table.loc["none", "logloss"] / 5, "L2 reduces the test log-loss by over 5 times compared to no penalty", "compare logloss")
    check(l1_true == table.loc["L1", "n_true"] and l1_true >= 4, "L1 finds on average at least 4 of the 6 informative features", "n_true of the L1 row")
    check(table.loc["L1", "n_coef"] > 12, "but it keeps many useless features (over 12): the selection is not clean", "n_coef of the L1 row")
done()

---
The commented solution is in `solutions/F_compare/F1_none_l2_l1_elasticnet.ipynb`. Open it after you pass the check, or after you have really tried.